# TPC_RP — Original Implementation

This notebook implements the **TypiClust with Random Projection (TPC_RP)** active learning algorithm
on CIFAR-10, following:

> Hacohen, G., Dekel, O., & Weinshall, D. (2022).  
> **Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets**.  
> *ICML 2022*. https://arxiv.org/abs/2202.02794

## Pipeline
1. Train SimCLR on the full CIFAR-10 training set (self-supervised)
2. Extract backbone features
3. Run TPC_RP active learning rounds
4. Evaluate with a linear probe after each round
5. Plot accuracy vs. label budget

## 0. Setup

In [ ]:
# Install dependencies (uncomment in Colab)
# !pip install -q -r ../requirements.txt

import sys
sys.path.insert(0, '..')

import numpy as np
import torch

from src.utils import set_seed, load_cifar10, extract_features, save_results, plot_accuracy_curve, log
from src.resnet import SimCLREncoder
from src.simclr import train_simclr
from src.active_learning import TypiClustSelector, run_active_learning_loop
from src.classifier import train_linear_probe, evaluate_linear_probe

SEED    = 42
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)
print(f'Device: {DEVICE}')

## 1. Hyperparameters

In [ ]:
# --- SimCLR ---
SIMCLR_EPOCHS  = 200   # increase for better representations (≥400 for publication quality)
SIMCLR_BATCH   = 256
SIMCLR_LR      = 3e-4
SIMCLR_TEMP    = 0.5

# --- Active learning ---
INITIAL_BUDGET = 10    # labelled samples before round 1
QUERY_BUDGET   = 10    # samples queried per round
N_ROUNDS       = 20    # total rounds  => max 10 + 19*10 = 200 labelled samples

# --- TPC_RP ---
PROJ_DIM = 64          # random projection target dimension
KNN_K    = 20          # neighbours for typicality

# --- Linear probe ---
PROBE_EPOCHS = 100
PROBE_LR     = 1e-3

## 2. Load Data

In [ ]:
train_dataset, test_dataset = load_cifar10(root='../data')
print(f'Train: {len(train_dataset):,}  |  Test: {len(test_dataset):,}')

## 3. SimCLR Pre-training

In [ ]:
encoder = SimCLREncoder(proj_dim=128, hidden_dim=512)
log('Starting SimCLR pre-training...')
encoder = train_simclr(
    dataset    = train_dataset,
    encoder    = encoder,
    epochs     = SIMCLR_EPOCHS,
    batch_size = SIMCLR_BATCH,
    lr         = SIMCLR_LR,
    temperature= SIMCLR_TEMP,
    device     = DEVICE,
)
torch.save(encoder.state_dict(), '../results/simclr_encoder.pt')
log('SimCLR pre-training complete.')

## 4. Feature Extraction

In [ ]:
log('Extracting train features...')
train_feats, train_labels = extract_features(train_dataset, encoder, device=DEVICE)
log('Extracting test features...')
test_feats, test_labels = extract_features(test_dataset, encoder, device=DEVICE)

np.save('../results/train_features.npy', train_feats)
np.save('../results/train_labels.npy',   train_labels)
np.save('../results/test_features.npy',  test_feats)
np.save('../results/test_labels.npy',    test_labels)

print(f'Train features: {train_feats.shape}  |  Test features: {test_feats.shape}')

## 5. Active Learning Loop (TPC_RP)

In [ ]:
selector = TypiClustSelector(proj_dim=PROJ_DIM, knn_k=KNN_K, seed=SEED)

def train_fn(labelled_idx, labels):
    return train_linear_probe(
        train_features = train_feats[labelled_idx],
        train_labels   = labels,
        epochs         = PROBE_EPOCHS,
        lr             = PROBE_LR,
        device         = DEVICE,
    )

def eval_fn(head):
    return evaluate_linear_probe(head, test_feats, test_labels, device=DEVICE)

history = run_active_learning_loop(
    features       = train_feats,
    labels         = train_labels,
    selector       = selector,
    initial_budget = INITIAL_BUDGET,
    query_budget   = QUERY_BUDGET,
    n_rounds       = N_ROUNDS,
    train_fn       = train_fn,
    eval_fn        = eval_fn,
    seed           = SEED,
)

save_results(history, '../results/tpcrp_original_history.json')

## 6. Results

In [ ]:
ax = plot_accuracy_curve(
    history['labelled_counts'],
    history['accuracies'],
    label='TPC_RP (original)',
    save_path='../plots/tpcrp_original.png',
)

import matplotlib.pyplot as plt
plt.show()

print('\nFinal accuracy: {:.2f}% with {:d} labelled samples'.format(
    history['accuracies'][-1] * 100,
    history['labelled_counts'][-1],
))